# Hash Maps

---

## What is a Hash Map?

A hash map (also called hash table or dictionary) is a data structure that stores **key-value pairs** and allows you to look up any value in **O(1) time** — no matter how many items are stored.

### The core idea
Instead of searching through all items to find something, a hash map **calculates exactly where to look** using a hash function:

```
key  →  hash function  →  index  →  stored value
```

### Why immutable keys?
Keys must be immutable (str, int, tuple) — NOT because you can't reassign variables, but because **the object itself must never change after creation**. If a key mutated, its hash would change, and Python could never find the stored value again. Lists are mutable (you can `.append()` to them in place), so they're forbidden as keys.

---
## Part 1 — Hash Function

A hash function takes a key and returns an index number. The simplest approach:
1. Loop through each character in the key
2. Get its ASCII value using `ord(char)` — e.g. `ord('a') = 97`
3. Sum them all up
4. Use `% size` to keep the result within array bounds

This guarantees the same key **always** maps to the same index.

In [6]:
# implement simple_hash(key, size)
# it should convert a string key into an array index
# use ord() to get character values, sum them, then mod by size

def simple_hash(key, size):
    sum = 0
    for char in key:
        ascii_value = ord(char)
        sum += ascii_value
    
    index = sum % size
    return index

# test it — run these and observe which keys collide
print(simple_hash("cat", 8))
print(simple_hash("dog", 8))
print(simple_hash("rat", 8))

# also try Python's built-in: print(hash("cat"))
print(hash("cat"))
print(hash("cat")%8)

0
2
7
-8406602147384626336
0


---
## Part 2 — Basic HashMap Class (no collision handling)

The internal structure is just a Python list of fixed size — each slot starts as `None`.

Methods to implement:
- `__init__` — create the array of empty slots
- `_hash` — internal method, converts key → index (use Python's `hash()` here, not your simple one)
- `set(key, value)` — compute index, store `(key, value)` at that slot
- `get(key)` — compute index, return the value at that slot (or None if empty)
- `display()` — print each slot with its index, useful for seeing what's inside

In [8]:
# implement the HashMap class
# __init__, _hash, set, get, display

class HashMap:
    def __init__(self, size=8):
        self.size = size
        self.array = [None] * size

    def _hash(self, key):
        l_integer = hash(key)  # large integer
        return l_integer % self.size
    
    def set(self, key, value):
        index = self._hash(key)
        self.array[index] = (key, value)

    def get(self, key):
        index = self._hash(key)
        if self.array[index] == None:
            return None
        else:
            return self.array[index][1]

    def display(self):
        for i, item in enumerate(self.array):
            print(i, item)
        

# test it
hm = HashMap()
hm.set("cat", 5)
hm.set("dog", 3)
hm.set("rat", 8)
hm.display()         # should show which slots are filled
print(hm.get("cat")) # should print 5

0 ('cat', 5)
1 None
2 None
3 None
4 ('rat', 8)
5 None
6 ('dog', 3)
7 None
5


---
## Part 3 — Collision Handling: Chaining

The HashMap above has a problem — if two keys hash to the same index, the second one silently **overwrites** the first. That's data loss.

**Chaining fix:** instead of each slot holding one pair, each slot holds a **list** of pairs. When a collision happens, just append to that list.

```
slot 3 → [ ("cat", 5), ("rat", 8) ]   ← both live here, no data lost
```

The `set` method now needs to:
1. Go to the right slot
2. Check if the key already exists in that list → update it
3. If not → append the new pair

The `get` method now needs to:
1. Go to the right slot
2. Loop through the list to find the matching key

In [ ]:
# implement HashMapWithChaining
# same methods as before but each bucket is a list, not None
# set must handle: update existing key OR append new pair
# get must loop through the bucket to find the right key

class HashMapWithChaining:
    def __init__(self, size=8):
        self.size = size
        self.array = [[] for _ in range(size)]

    def _hash(self, key):
        l_integer = hash(key)  # large integer
        return l_integer % self.size
    
    def set(self, key, value):
        index = self._hash(key)
        for i, (k, v) in enumerate(self.array[index]):
            if k == key:
                self.array[index][i] = (key, value)
                return

        self.array[index].append((key, value))

    def get(self, key):
        index = self._hash(key)
        for (k, v) in self.array[index]:
            if k == key:
                return v
        return None

    def display(self):
        for i, item in enumerate(self.array):
            print(i, item)
        

# test with small size to force collisions
hm2 = HashMapWithChaining(size=4)
hm2.set("cat", 5)
hm2.set("dog", 3)
hm2.set("rat", 8)
hm2.set("bat", 2)
hm2.display()         # some slots should have multiple pairs
print(hm2.get("rat")) # should still return 8

0 [('cat', 5), ('rat', 8)]
1 [('bat', 2)]
2 [('dog', 3)]
3 []
8


In [ ]:
arr = [[]] * 4     # this creates 4 references that point to the same list rather than 4 empty lists
arr[0].append("cat")  # to create list of empty list, we can use list comprehension like in above class
print(arr)

[['cat'], ['cat'], ['cat'], ['cat']]


---
## Part 4 — Big-O Analysis

| Operation | Average Case | Worst Case (all keys collide) |
|-----------|-------------|-------------------------------|
| `set` | **O(1)** | O(n) |
| `get` | **O(1)** | O(n) |
| `delete` | **O(1)** | O(n) |
| Space | **O(n)** | O(n) |

Worst case only happens with a terrible hash function. Python's `hash()` makes collisions essentially impossible in practice.

**Space is O(n)** — one slot per stored item.

---
## Part 5 — Python dict in Action

Python's `dict` is a hash map. Use it to count word frequency — this exact pattern is the foundation of NLP (TF-IDF, tokenization, vocabulary building).

In [14]:
# count how many times each word appears in this text
# use only a plain dict — no Counter, no collections
# hint: dict.get(key, default) is useful here

text = "the cat sat on the mat the cat"

# your code here
words = text.split(" ")
word_count = {}
for word in words:
    word_count[word] = word_count.get(word, 0) + 1

print(word_count)

# expected: {'the': 3, 'cat': 2, 'sat': 1, 'on': 1, 'mat': 1}

{'the': 3, 'cat': 2, 'sat': 1, 'on': 1, 'mat': 1}


### use of dict.get(key, default)
It tries to get the value for key — but if the key doesn't exist, it returns default instead of crashing.

d = {"cat": 5}

d["dog"]           # ❌ KeyError — "dog" doesn't exist  
d.get("dog")       # → None  (no crash).  
d.get("dog", 0)    # → 0     (returns default instead)  
d.get("cat", 0)    # → 5     (key exists, returns actual value)  
So for word counting — on the first time you see a word, get returns 0. You add 1 to it and store.  it. Next time you see the same word, get returns whatever count is already there, and you add 1 again. 

---
## Part 6 — Real-World Use Cases

| Use Case | How hash maps are used |
|----------|------------------------|
| **Database indexing** | Maps a value to its row location — O(1) lookup instead of a full table scan |
| **Caching (Redis)** | Stores the result of an expensive computation by key — next request is instant |
| **NLP / word frequency** | Counts word occurrences — direct foundation of TF-IDF |
| **RAG systems** | Maps document IDs to their embeddings/metadata |
| **Python itself** | Every object's attributes live in `obj.__dict__` — a hash map |


### Database Indexing
Without an index, finding a user in a database of 10 million rows means checking every row — O(n).
With an index, the database builds a hash map: `user_id → exact row location`.
This is why adding an index to a column makes queries dramatically faster.

### Caching (Redis)
Say your API calculates a user's total spending — queries the database, runs calculations, takes 200ms.
Redis stores the result: `"user_123_total" → 4500`. Next request? Instant lookup, no recalculation.
You store the **result** for a specific input, not the function itself.

```python
# first request — expensive
result = calculate_total_spending("user_123")  # 200ms
redis.set("user_123_total", result)

# second request — instant
redis.get("user_123_total")  # O(1), no recalculation  
```

### NLP / Word Frequency  
Counting how often each word appears in a document is the foundation of how search engines
and AI models understand text. TF-IDF is built directly on word frequency
hash maps.

### RAG Systems  
Each document chunk gets converted to an embedding (a list of ~1500 numbers capturing meaning).
These are stored as document_id → embedding. When a user asks a question, you convert it to
an embedding and find the most similar stored ones — the hash map gives O(1) access to any
document's embedding by ID.


embeddings = {  
    "doc_001": [0.23, 0.87, -0.12, ...],  # chunk about Python  
    "doc_002": [0.91, 0.04, 0.67, ...],   # chunk about databases  
}  
### Python Itself
Every object you create has a __dict__. When you do account.balance, Python looks up
"balance" in account.__dict__ — a hash map lookup. Attribute access is O(1) no matter
how many attributes an object has.




---
## Part 7 — Reflection

Write answers in your own words:

1. Why is lookup O(1) in a hash map but O(n) in a list?  
=> Because in a hash map we search through the key, we pass the key to a hash function and get an index, at that index the data is stored, so we instantly fetch the data. But in list we do not know wehre to look for the data directly so we loop through all the items to search for that data.  

2. What happens during a collision, and how does chaining fix it?  
=> During collision, two keys get hashed into a single index, that means both share the same index number so to fix this we use chaining tha allows both keys to live inside the list together. There is another approach called open addressing(probing) where free slot right of that index slot gets assigned to the conflicting key.  
3. Why can't a list be a dict key, but a tuple can?  
=> To be a key, the object has to be immutable(i.e it cannot be changed once set), we can use methods like append, remove, to change the list object but the same thing doesn't apply for tuple.  
4. You have 1 million user records. Someone hits your API and you need to check if a username already exists. List or dict — which do you use and why?  
=> I would use a dict because in dict we can directly query the username in o(1) time, but in list on average it would take O(n) time. 

---
## LeetCode

Try each problem for at least 20 minutes before asking for hints.

- [ ] Two Sum (Easy)
- [ ] Contains Duplicate (Easy)
- [ ] Valid Anagram (Easy)

In [16]:
# TWO SUM
# given a list of numbers and a target, return the indices of two numbers that add up to target
# example: nums=[2,7,11,15], target=9 → [0, 1]  (because 2+7=9)
# constraint: you may not use the same element twice
# think: for each number, what is the value you still NEED to reach the target?

def two_sum(nums, target):
    seen = {}
    for i, num in enumerate(nums):
        need = target - num
        if need in seen:
            return seen[need], i
        seen[num] = i


print(two_sum([2, 7, 11, 15], 9))   # [0, 1]
print(two_sum([3, 2, 4], 6))        # [1, 2]

(0, 1)
(1, 2)


## `in` Operator — Speed by Data Structure

```python
# LIST — O(n), scans every item
3 in [1, 2, 3, 4]

# DICT — O(1), checks keys only
"cat" in {"cat": 5}      # True
"dog" in {"cat": 5}      # False
5 in {"cat": 5}          # False — checks keys, not values

# SET — O(1), a hash map with no values, just keys
3 in {1, 2, 3}

# STRING — O(n)
"cat" in "the cat sat"   # True
```

**Key rule:** use `dict` or `set` when you need fast membership checks.
If you use a list, `x in list` is O(n) — your whole solution becomes O(n²).

### Why set is O(1)
A set is a hash map with no values — only keys.
`3 in {1, 2, 3}` hashes `3`, gets an index, checks that slot directly.
Same mechanism as dict, just without storing a value alongside the key.


In [ ]:
# CONTAINS DUPLICATE
# return True if any value appears at least twice in the list
# example: [1,2,3,1] → True
# think: what data structure tells you instantly if you've seen something before?

def contains_duplicate(nums):
    seen = {}
    for num in nums:
        if num in seen:
            return True
        seen[num] = True
    return False

def contains_duplicate_using_set(nums):
    seen = set()
    for num in nums:
        if num in seen:
            return True
        seen.add(num)
    return False

print(contains_duplicate([1, 2, 3, 1]))   # True
print(contains_duplicate([1, 2, 3, 4]))   # False

True
False


In [20]:
# VALID ANAGRAM
# return True if t is an anagram of s (same letters, different order)
# example: s="anagram", t="nagaram" → True
# think: two words are anagrams if they have exactly the same character frequencies

def is_anagram(s, t):
    s_dict = {}
    t_dict = {}
    for char in s:
        s_dict[char] = s_dict.get(char, 0) + 1
    
    for char in t:
        t_dict[char] = t_dict.get(char, 0) + 1

    # if s_dict == t_dict:
    #     return True
    # return False

    return s_dict == t_dict

print(is_anagram("anagram", "nagaram"))   # True
print(is_anagram("rat", "car"))           # False

True
False
